# ARC_ATLAS v4 Small-Lesion Runbook

This notebook now runs the current small-lesion trainer with an explicit proposal branch added on top of the dense whole-brain mask head.

## Active Small-Lesion Measures
- Source-balanced case sampling.
- Small-lesion-aware case grouping: `1-99`, `100-999`, `1000-9999`, `10000+` voxels.
- Soft lesion-size curriculum during the early epochs:
  - starts tiny-heavy
  - then anneals back toward the mixed-dataset policy
- Targeted tiny/small component centering:
  - when tiny or small lesions exist, patch centers are pulled toward those components with tight jitter
  - the old aggressive full component-aware sampler remains off
- Proposal-aware localization branch:
  - `center_heatmap` head predicts lesion-center likelihood
  - whole-brain validation reports proposal recall at top-K candidate centers
  - center diagnostics now separate raw peak score from thresholded candidate count
- Controlled auxiliary supervision:
  - extra auxiliary heads are off for this ablation so the dense mask head and center head can be judged cleanly
- Late ATLAS-focused fine-tuning phase:
  - after the mixed phase, sampler source mass shifts toward ATLAS while keeping some non-ATLAS exposure
- Generic spatial/intensity augmentation on all patches.
- Lower-learning-rate, long-horizon training schedule with regularization.
- Whole-brain validation every epoch with:
  - soft Dice
  - raw hard Dice before any masking
  - center proposal recall at top-K
  - proposal size-seed accuracy
  - brain-mask-clipped hard Dice as a diagnostic only
  - threshold sweep
  - per-source metrics
  - per-case CSV diagnostics

## Intentionally Disabled Right Now
- Brain-mask clipping in the main postprocessing path.
- Component-scoring postprocessing in the main path.
- Lesion insertion augmentation.
- Symmetric flip-channel input.
- TopK hard-voxel loss.
- Full component-aware patch sampling.

## Main Artifacts To Watch
- `callbacks/training_log.csv`
- `callbacks/batch_metrics.csv`
- `callbacks/whole_val_summary.jsonl`
- `callbacks/whole_val_epoch_XXXX.csv`
- `callbacks/sampling_schedule.jsonl`
- `callbacks/diagnostics/training_summary.md`




In [1]:
# test

In [ ]:
from pathlib import Path
import importlib.util
import shutil
import time
import traceback

# --------- Paths and module loading ---------
PROJECT_ROOT = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()
RUN_ROOT = PROJECT_ROOT
SRC = PROJECT_ROOT / "src" / "training_v2_smalllesion.py"

TRAIN_DIR = Path("/home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/ARC_ATLAS_Train_v4/data/splits/90_10_random/train")
TRAIN_T1 = TRAIN_DIR / "t1"
TRAIN_MASKS = TRAIN_DIR / "masks"

if not SRC.exists():
    raise FileNotFoundError(f"Training module not found: {SRC}")
if not TRAIN_DIR.exists():
    raise FileNotFoundError(f"Training data dir not found: {TRAIN_DIR}. Run ARC_ATLAS_TrainPrep_v4.ipynb first.")
if not TRAIN_T1.exists() or not TRAIN_MASKS.exists():
    raise FileNotFoundError(f"Expected subfolders missing under {TRAIN_DIR}: t1/ and masks/")

spec = importlib.util.spec_from_file_location("seg", SRC)
if spec is None or spec.loader is None:
    raise RuntimeError(f"Could not load module spec from {SRC}")
seg = importlib.util.module_from_spec(spec)
spec.loader.exec_module(seg)

# --------- Hyperparameters ---------
INPUT_SHAPE = (208, 240, 208, 1)
PATCH_SIZE = (208, 240, 208)
PATCHES_PER_CASE = 2
EPOCH_STEPS = 256
FIT_VERBOSE = 2
MEMORY_LOGS_ENABLED = False
DIAGNOSTICS_ENABLED = True
BATCH_LOG_EVERY_N_STEPS = 1
DIAGNOSTICS_COMPARE_POSTPROC = True
TOTAL_EPOCHS = 200
INITIAL_EPOCH = 0

BASE_FILTERS = 4
SAM_HEADS = 2
BATCH_SIZE = 1
VAL_SPLIT = 0.10
DROPOUT_RATE = 0.35
L2_REG = 3e-4

AUG_INTENSITY = 0.45
ROTATION_RANGE = 25
SMALL_LESION_THRESHOLD = 6000
SYNTHETIC_LESION_PROB = 0.6

INITIAL_LR = 5e-5
MIN_LR = 1e-6
WARMUP_EPOCHS = 10
COSINE_FIRST_CYCLE_EPOCHS = 100
COSINE_T_MUL = 1.0
COSINE_M_MUL = 1.0
SWA_EPOCHS = 0
SWA_LR_MULT = None

DICE_WEIGHT = 0.45
BOUNDARY_WEIGHT = 0.30
BCE_WEIGHT = 0.20
VOLUME_RATIO_WEIGHT = 0.05
BOUNDARY_WARMUP_DICE = 0.4
BOUNDARY_WARMUP_BOUNDARY = 0.6
BOUNDARY_RAMP_EPOCHS = 1

FOCAL_TVERSKY_WEIGHT = 0.0
TVERSKY_ALPHA = 0.7
TVERSKY_BETA = 0.3
FOCAL_TVERSKY_GAMMA = 1.5

SIZE_BUCKET_PROBS = (0.45, 0.25, 0.15, 0.10, 0.05)
PATCH_FG_PROB_BY_BIN = (0.98, 0.95, 0.85, 0.70)
SOURCE_BALANCED_SAMPLING = True
OUTPUT_BIAS_INIT_PROB = 0.015
USE_SYMMETRIC_FLIP_CHANNEL = False
CASE_SIZE_BINS = (100, 1000, 10000)
CASE_SIZE_GROUP_PROBS = (0.45, 0.25, 0.15, 0.10)
CASE_NONE_PROB = 0.05
USE_COMPONENT_AWARE_PATCH_SAMPLING = False
USE_TINY_COMPONENT_CENTERING = True
TINY_COMPONENT_CENTER_PROB = 0.95
SMALL_COMPONENT_CENTER_PROB = 0.85
TINY_COMPONENT_MAX_JITTER = 2
SMALL_COMPONENT_MAX_JITTER = 4
MSL_COMPONENT_THRESHOLDS = (100, 1000, 10000)
USE_CENTER_HEATMAP_HEAD = False
USE_SIZE_HEAD = False
CENTER_HEATMAP_SIGMA = 4.0
CENTER_POSITIVE_WEIGHT = 10.0
AUX_CENTER_WEIGHT = 0.12
AUX_SIZE_WEIGHT = 0.05
SIZE_HEAD_CLASS_WEIGHTS = (0.02, 4.0, 2.5, 1.0, 0.6)
CENTER_TOPK_VALUES = (1, 3, 5, 10, 20)
CENTER_MATCH_RADIUS = 6.0
CENTER_NMS_RADIUS = 6
CENTER_MIN_CONFIDENCE = 0.01
CENTER_HEAD_BIAS_INIT_PROB = 0.01
CENTER_LOSS_GAMMA = 2.0
CENTER_LOSS_BETA = 4.0
USE_AUX_MSL_HEAD = False
USE_AUX_DBL_HEAD = False
AUX_MSL_WEIGHT = 0.00
AUX_DBL_WEIGHT = 0.00
AUX_MSL_CLASS_WEIGHTS = (0.02, 4.0, 2.5, 1.0, 0.6)
AUX_DBL_CLASS_WEIGHTS = (0.02, 1.15, 1.0)
USE_SIZE_CURRICULUM = True
CURRICULUM_EPOCHS = 12
CURRICULUM_START_CASE_GROUP_PROBS = (0.70, 0.20, 0.07, 0.03)
CURRICULUM_START_PATCH_FG_PROB_BY_BIN = (0.995, 0.99, 0.92, 0.78)
CURRICULUM_START_CASE_NONE_PROB = 0.02
USE_ATLAS_FINE_TUNE = True
ATLAS_FINE_TUNE_START_EPOCH = 70
ATLAS_FINE_TUNE_SOURCE_PREFIXES = ("ATLAS",)
ATLAS_FINE_TUNE_SOURCE_MASS = 0.70
ATLAS_FINE_TUNE_CASE_GROUP_PROBS = (0.60, 0.25, 0.10, 0.05)
ATLAS_FINE_TUNE_PATCH_FG_PROB_BY_BIN = (0.995, 0.985, 0.94, 0.82)
ATLAS_FINE_TUNE_CASE_NONE_PROB = 0.02
TOPK_VOXEL_FRACTION = 0.00
TOPK_WEIGHT = 0.00
LESION_INSERTION_PROB = 0.00
LESION_INSERTION_MAX_COMPONENT_VOXELS = 1000
USE_BRAINMASK_POSTPROC = False
USE_COMPONENT_SCORING_POSTPROC = False
GROUPED_CV_FOLDS = 3
EXTERNAL_VAL_DIR = None
EXTERNAL_VAL_MANIFEST = None

# Full-image patch extraction controls
LOAD_FULL_IMAGE_FOR_PATCHING = True
FULL_RES_TARGET_SHAPE = (208, 240, 208)
WHOLE_BRAIN_VAL_ENABLED = True
WHOLE_BRAIN_VAL_EVERY_N_EPOCHS = 1
WHOLE_BRAIN_VAL_MAX_CASES = None
WHOLE_BRAIN_VAL_TTA = False
PATCH_SAMPLING_STRATEGY = "random"
HEMISPHERE_AXIS = 2
HEMISPHERE_BALANCED = True

# --------- Per-run artifact directories ---------
RUN_ID = time.strftime("%Y%m%d_%H%M%S")
RUN_DIR = RUN_ROOT / "runs" / RUN_ID
MODEL_DIR = RUN_DIR / "models"
CALLBACKS_DIR = RUN_DIR / "callbacks"
for d in (MODEL_DIR, CALLBACKS_DIR):
    d.mkdir(parents=True, exist_ok=True)

print("Using training module:", SRC)
print("Training data:", TRAIN_DIR)
print("Run dir:", RUN_DIR)
ACTIVE_SMALL_LESION_MEASURES = {
    "source_balanced_sampling": SOURCE_BALANCED_SAMPLING,
    "case_size_bins": CASE_SIZE_BINS,
    "case_size_group_probs": CASE_SIZE_GROUP_PROBS,
    "case_none_prob": CASE_NONE_PROB,
    "patch_fg_prob_by_bin": PATCH_FG_PROB_BY_BIN,
    "component_aware_patch_sampling": USE_COMPONENT_AWARE_PATCH_SAMPLING,
    "tiny_component_centering": USE_TINY_COMPONENT_CENTERING,
    "center_heatmap_head": USE_CENTER_HEATMAP_HEAD,
    "size_head": USE_SIZE_HEAD,
    "center_heatmap_sigma": CENTER_HEATMAP_SIGMA,
    "center_head_bias_init_prob": CENTER_HEAD_BIAS_INIT_PROB,
    "center_loss_gamma": CENTER_LOSS_GAMMA,
    "center_loss_beta": CENTER_LOSS_BETA,
    "center_topk_values": CENTER_TOPK_VALUES,
    "center_match_radius": CENTER_MATCH_RADIUS,
    "size_head_class_weights": SIZE_HEAD_CLASS_WEIGHTS,
    "aux_msl_class_weights": AUX_MSL_CLASS_WEIGHTS,
    "aux_dbl_class_weights": AUX_DBL_CLASS_WEIGHTS,
    "size_curriculum": USE_SIZE_CURRICULUM,
    "curriculum_epochs": CURRICULUM_EPOCHS,
    "curriculum_start_case_group_probs": CURRICULUM_START_CASE_GROUP_PROBS,
    "curriculum_start_patch_fg_prob_by_bin": CURRICULUM_START_PATCH_FG_PROB_BY_BIN,
    "atlas_fine_tune": USE_ATLAS_FINE_TUNE,
    "atlas_fine_tune_start_epoch": ATLAS_FINE_TUNE_START_EPOCH,
    "atlas_fine_tune_source_mass": ATLAS_FINE_TUNE_SOURCE_MASS,
    "brainmask_postproc": USE_BRAINMASK_POSTPROC,
    "component_scoring_postproc": USE_COMPONENT_SCORING_POSTPROC,
    "aux_msl_head": USE_AUX_MSL_HEAD,
    "aux_dbl_head": USE_AUX_DBL_HEAD,
    "topk_weight": TOPK_WEIGHT,
    "lesion_insertion_prob": LESION_INSERTION_PROB,
    "symmetric_flip_channel": USE_SYMMETRIC_FLIP_CHANNEL,
    "initial_lr": INITIAL_LR,
    "total_epochs": TOTAL_EPOCHS,
    "dropout_rate": DROPOUT_RATE,
}
print("Active small-lesion measures:")
for k, v in ACTIVE_SMALL_LESION_MEASURES.items():
    print(f" - {k}: {v}")

# --------- Train tiny-lesion-aware ablation run ---------
try:
    history = seg.train_dynamic_model(
        DATA_DIR=TRAIN_DIR,
        IMAGES_DIR=TRAIN_T1,
        MASKS_DIR=TRAIN_MASKS,
        MODEL_DIR=MODEL_DIR,
        CALLBACKS_DIR=CALLBACKS_DIR,
        INPUT_SHAPE=INPUT_SHAPE,
        BASE_FILTERS=BASE_FILTERS,
        SAM_HEADS=SAM_HEADS,
        BATCH_SIZE=BATCH_SIZE,
        DROPOUT_RATE=DROPOUT_RATE,
        L2_REG=L2_REG,
        PATCH_SIZE=PATCH_SIZE,
        PATCHES_PER_CASE=PATCHES_PER_CASE,
        EPOCH_STEPS=EPOCH_STEPS,
        FIT_VERBOSE=FIT_VERBOSE,
        MEMORY_LOGS_ENABLED=MEMORY_LOGS_ENABLED,
        DIAGNOSTICS_ENABLED=DIAGNOSTICS_ENABLED,
        BATCH_LOG_EVERY_N_STEPS=BATCH_LOG_EVERY_N_STEPS,
        DIAGNOSTICS_COMPARE_POSTPROC=DIAGNOSTICS_COMPARE_POSTPROC,
        TOTAL_EPOCHS=TOTAL_EPOCHS,
        INITIAL_EPOCH=INITIAL_EPOCH,
        RESAMPLE_TO_TARGET=False,
        AUGMENTATION_INTENSITY=AUG_INTENSITY,
        ROTATION_RANGE=ROTATION_RANGE,
        SMALL_LESION_THRESHOLD=SMALL_LESION_THRESHOLD,
        SYNTHETIC_LESION_PROB=SYNTHETIC_LESION_PROB,
        INITIAL_LR=INITIAL_LR,
        MIN_LR=MIN_LR,
        WARMUP_EPOCHS=WARMUP_EPOCHS,
        COSINE_FIRST_CYCLE_EPOCHS=COSINE_FIRST_CYCLE_EPOCHS,
        COSINE_T_MUL=COSINE_T_MUL,
        COSINE_M_MUL=COSINE_M_MUL,
        COSINE_MIN_LR_MULT=0.1,
        SWA_EPOCHS=SWA_EPOCHS,
        SWA_LR_MULT=SWA_LR_MULT,
        DICE_WEIGHT=DICE_WEIGHT,
        BOUNDARY_WEIGHT=BOUNDARY_WEIGHT,
        BCE_WEIGHT=BCE_WEIGHT,
        VOLUME_RATIO_WEIGHT=VOLUME_RATIO_WEIGHT,
        DICE_LOSS_WEIGHT=0.4,
        BOUNDARY_LOSS_WEIGHT=0.6,
        BOUNDARY_WARMUP_DICE=BOUNDARY_WARMUP_DICE,
        BOUNDARY_WARMUP_BOUNDARY=BOUNDARY_WARMUP_BOUNDARY,
        BOUNDARY_RAMP_EPOCHS=BOUNDARY_RAMP_EPOCHS,
        FOCAL_TVERSKY_WEIGHT=FOCAL_TVERSKY_WEIGHT,
        TVERSKY_ALPHA=TVERSKY_ALPHA,
        TVERSKY_BETA=TVERSKY_BETA,
        FOCAL_TVERSKY_GAMMA=FOCAL_TVERSKY_GAMMA,
        SIZE_BUCKET_PROBS=SIZE_BUCKET_PROBS,
        PATCH_FG_PROB_BY_BIN=PATCH_FG_PROB_BY_BIN,
        SOURCE_BALANCED_SAMPLING=SOURCE_BALANCED_SAMPLING,
        OUTPUT_BIAS_INIT_PROB=OUTPUT_BIAS_INIT_PROB,
        USE_SYMMETRIC_FLIP_CHANNEL=USE_SYMMETRIC_FLIP_CHANNEL,
        CASE_SIZE_BINS=CASE_SIZE_BINS,
        CASE_SIZE_GROUP_PROBS=CASE_SIZE_GROUP_PROBS,
        CASE_NONE_PROB=CASE_NONE_PROB,
        USE_COMPONENT_AWARE_PATCH_SAMPLING=USE_COMPONENT_AWARE_PATCH_SAMPLING,
        USE_TINY_COMPONENT_CENTERING=USE_TINY_COMPONENT_CENTERING,
        TINY_COMPONENT_CENTER_PROB=TINY_COMPONENT_CENTER_PROB,
        SMALL_COMPONENT_CENTER_PROB=SMALL_COMPONENT_CENTER_PROB,
        TINY_COMPONENT_MAX_JITTER=TINY_COMPONENT_MAX_JITTER,
        SMALL_COMPONENT_MAX_JITTER=SMALL_COMPONENT_MAX_JITTER,
        MSL_COMPONENT_THRESHOLDS=MSL_COMPONENT_THRESHOLDS,
        USE_CENTER_HEATMAP_HEAD=USE_CENTER_HEATMAP_HEAD,
        USE_SIZE_HEAD=USE_SIZE_HEAD,
        CENTER_HEATMAP_SIGMA=CENTER_HEATMAP_SIGMA,
        CENTER_POSITIVE_WEIGHT=CENTER_POSITIVE_WEIGHT,
        AUX_CENTER_WEIGHT=AUX_CENTER_WEIGHT,
        AUX_SIZE_WEIGHT=AUX_SIZE_WEIGHT,
        SIZE_HEAD_CLASS_WEIGHTS=SIZE_HEAD_CLASS_WEIGHTS,
        CENTER_TOPK_VALUES=CENTER_TOPK_VALUES,
        CENTER_MATCH_RADIUS=CENTER_MATCH_RADIUS,
        CENTER_NMS_RADIUS=CENTER_NMS_RADIUS,
        CENTER_MIN_CONFIDENCE=CENTER_MIN_CONFIDENCE,
        CENTER_HEAD_BIAS_INIT_PROB=CENTER_HEAD_BIAS_INIT_PROB,
        CENTER_LOSS_GAMMA=CENTER_LOSS_GAMMA,
        CENTER_LOSS_BETA=CENTER_LOSS_BETA,
        USE_AUX_MSL_HEAD=USE_AUX_MSL_HEAD,
        USE_AUX_DBL_HEAD=USE_AUX_DBL_HEAD,
        AUX_MSL_WEIGHT=AUX_MSL_WEIGHT,
        AUX_DBL_WEIGHT=AUX_DBL_WEIGHT,
        AUX_MSL_CLASS_WEIGHTS=AUX_MSL_CLASS_WEIGHTS,
        AUX_DBL_CLASS_WEIGHTS=AUX_DBL_CLASS_WEIGHTS,
        USE_SIZE_CURRICULUM=USE_SIZE_CURRICULUM,
        CURRICULUM_EPOCHS=CURRICULUM_EPOCHS,
        CURRICULUM_START_CASE_GROUP_PROBS=CURRICULUM_START_CASE_GROUP_PROBS,
        CURRICULUM_START_PATCH_FG_PROB_BY_BIN=CURRICULUM_START_PATCH_FG_PROB_BY_BIN,
        CURRICULUM_START_CASE_NONE_PROB=CURRICULUM_START_CASE_NONE_PROB,
        USE_ATLAS_FINE_TUNE=USE_ATLAS_FINE_TUNE,
        ATLAS_FINE_TUNE_START_EPOCH=ATLAS_FINE_TUNE_START_EPOCH,
        ATLAS_FINE_TUNE_SOURCE_PREFIXES=ATLAS_FINE_TUNE_SOURCE_PREFIXES,
        ATLAS_FINE_TUNE_SOURCE_MASS=ATLAS_FINE_TUNE_SOURCE_MASS,
        ATLAS_FINE_TUNE_CASE_GROUP_PROBS=ATLAS_FINE_TUNE_CASE_GROUP_PROBS,
        ATLAS_FINE_TUNE_PATCH_FG_PROB_BY_BIN=ATLAS_FINE_TUNE_PATCH_FG_PROB_BY_BIN,
        ATLAS_FINE_TUNE_CASE_NONE_PROB=ATLAS_FINE_TUNE_CASE_NONE_PROB,
        TOPK_VOXEL_FRACTION=TOPK_VOXEL_FRACTION,
        TOPK_WEIGHT=TOPK_WEIGHT,
        LESION_INSERTION_PROB=LESION_INSERTION_PROB,
        LESION_INSERTION_MAX_COMPONENT_VOXELS=LESION_INSERTION_MAX_COMPONENT_VOXELS,
        USE_BRAINMASK_POSTPROC=USE_BRAINMASK_POSTPROC,
        USE_COMPONENT_SCORING_POSTPROC=USE_COMPONENT_SCORING_POSTPROC,
        GROUPED_CV_FOLDS=GROUPED_CV_FOLDS,
        EXTERNAL_VAL_DIR=EXTERNAL_VAL_DIR,
        EXTERNAL_VAL_MANIFEST=EXTERNAL_VAL_MANIFEST,
        LOAD_FULL_IMAGE_FOR_PATCHING=LOAD_FULL_IMAGE_FOR_PATCHING,
        FULL_RES_TARGET_SHAPE=FULL_RES_TARGET_SHAPE,
        WHOLE_BRAIN_VAL_ENABLED=WHOLE_BRAIN_VAL_ENABLED,
        WHOLE_BRAIN_VAL_EVERY_N_EPOCHS=WHOLE_BRAIN_VAL_EVERY_N_EPOCHS,
        WHOLE_BRAIN_VAL_MAX_CASES=WHOLE_BRAIN_VAL_MAX_CASES,
        WHOLE_BRAIN_VAL_TTA=WHOLE_BRAIN_VAL_TTA,
        PATCH_SAMPLING_STRATEGY=PATCH_SAMPLING_STRATEGY,
        HEMISPHERE_AXIS=HEMISPHERE_AXIS,
        HEMISPHERE_BALANCED=HEMISPHERE_BALANCED,
        DIFF_AWARE_ENABLED=True,
        DIFF_EMA_LAMBDA=0.8,
        DIFF_BETA=1.5,
        VALIDATION_SPLIT=VAL_SPLIT,
        LOAD_WEIGHTS_FROM=None,
        RESUME_FROM_LATEST=False,
    )
    print("Training complete. Keys:", list(getattr(history, "history", {}).keys()))
    print("Artifacts saved to", RUN_DIR)
except Exception:
    traceback.print_exc()
    raise

# Convenience: mark this run as latest
latest_link = RUN_ROOT / "runs" / "latest"
if latest_link.exists() or latest_link.is_symlink():
    latest_link.unlink()
latest_link.symlink_to(RUN_DIR, target_is_directory=True)

best_src = CALLBACKS_DIR / "best_model_dynamic.weights.h5"
if best_src.exists():
    best_copy = RUN_ROOT / "runs" / "latest_best.weights.h5"
    shutil.copy2(best_src, best_copy)
    print("Saved best copy ->", best_copy)





2026-04-07 11:35:09.750977: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


Visible GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')]
Mixed precision policy: <DTypePolicy "float32">
INFO:tensorflow:Using MirroredStrategy with devices ('/job:localhost/replica:0/task:0/device:GPU:0', '/job:localhost/replica:0/task:0/device:GPU:1')


I0000 00:00:1775583311.927724 3129718 gpu_process_state.cc:208] Using CUDA malloc Async allocator for GPU: 0
I0000 00:00:1775583311.928769 3129718 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 22148 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4090, pci bus id: 0000:41:00.0, compute capability: 8.9
I0000 00:00:1775583311.929093 3129718 gpu_process_state.cc:208] Using CUDA malloc Async allocator for GPU: 1
I0000 00:00:1775583311.930070 3129718 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 22122 MB memory:  -> device: 1, name: NVIDIA GeForce RTX 4090, pci bus id: 0000:61:00.0, compute capability: 8.9
2026-04-07 11:35:11,997 - SmartSOTA_Dynamic - INFO - ✅ All imports successful
2026-04-07 11:35:11,997 - SmartSOTA_Dynamic - INFO - TensorFlow eager execution: True
2026-04-07 11:35:11,997 - SmartSOTA_Dynamic - INFO - Environment verified:
- Python 3.10.18 (main, Jun  5 2025, 13:14:17) [GCC 11.2.0]
- TensorFlo

Strategy: MirroredStrategy
Using training module: /home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/ARC_ATLAS_Train_v4/src/training_v2_smalllesion.py
Training data: /home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/ARC_ATLAS_Train_v4/data/splits/90_10_random/train
Run dir: /home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/ARC_ATLAS_Train_v4/runs/20260407_113512
Active small-lesion measures:
 - source_balanced_sampling: True
 - case_size_bins: (100, 1000, 10000)
 - case_size_group_probs: (0.45, 0.25, 0.15, 0.1)
 - case_none_prob: 0.05
 - patch_fg_prob_by_bin: (0.98, 0.95, 0.85, 0.7)
 - component_aware_patch_sampling: False
 - tiny_component_centering: True
 - center_heatmap_head: False
 - size_head: False
 - center_heatmap_sigma: 4.0
 - center_head_bias_init_prob: 0.01
 - center_loss_gamma: 2.0
 - center_loss_beta: 4.0
 - center_topk_values: (1, 3, 5, 10, 20)
 - center_match_radius: 6.0
 - size_head_class_weights: (0.02, 4.0, 2.5, 1.0, 0.6)
 - aux_msl_class_weights: (0.02, 4.0, 2.5, 1.0, 0

2026-04-07 11:35:13,220 - SmartSOTA_Dynamic - INFO - Model built: 697,909 parameters
2026-04-07 11:35:13,220 - SmartSOTA_Dynamic - INFO - 📚 Loading dataset (flex loader for T1w volumes)…
2026-04-07 11:35:13,221 - SmartSOTA_Dynamic - INFO - 📄 Using manifest-defined pairs from /home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/ARC_ATLAS_Train_v4/data/splits/90_10_random/train/manifest.csv
2026-04-07 11:36:57,841 - SmartSOTA_Dynamic - INFO - Manifest composition: {'ARC-combined-t1-raw-ab0d1794': 190, 'ATLAS-Images-f0d7431e': 582, 'Approx-Numeracy-Processed': 94}
2026-04-07 11:36:57,842 - SmartSOTA_Dynamic - INFO - 📊 Created 866 image–mask pairs from manifest
2026-04-07 11:36:57,842 - SmartSOTA_Dynamic - INFO - 🧠 Lesion presence: 99.65%
2026-04-07 11:43:11,377 - SmartSOTA_Dynamic - INFO - 🧮 Dataset split (stratified_source): Train=779 (90.0%), Validation=87 (10.0%)
2026-04-07 11:43:11,378 - SmartSOTA_Dynamic - INFO - 🧩 Stratification groups: {'ARC-combined-t1-raw-ab0d1794': 190, 'ATLAS-Image

...

Epoch 12/200


2026-04-07 13:30:10,247 - SmartSOTA_Dynamic - INFO - Whole-brain val progress: 8/87 cases
2026-04-07 13:30:27,014 - SmartSOTA_Dynamic - INFO - Whole-brain val progress: 16/87 cases
2026-04-07 13:30:43,449 - SmartSOTA_Dynamic - INFO - Whole-brain val progress: 24/87 cases
2026-04-07 13:31:00,156 - SmartSOTA_Dynamic - INFO - Whole-brain val progress: 32/87 cases
2026-04-07 13:31:17,484 - SmartSOTA_Dynamic - INFO - Whole-brain val progress: 40/87 cases
2026-04-07 13:31:33,555 - SmartSOTA_Dynamic - INFO - Whole-brain val progress: 48/87 cases
2026-04-07 13:31:48,612 - SmartSOTA_Dynamic - INFO - Whole-brain val progress: 56/87 cases
2026-04-07 13:32:03,183 - SmartSOTA_Dynamic - INFO - Whole-brain val progress: 64/87 cases
2026-04-07 13:32:17,768 - SmartSOTA_Dynamic - INFO - Whole-brain val progress: 72/87 cases
2026-04-07 13:32:32,287 - SmartSOTA_Dynamic - INFO - Whole-brain val progress: 80/87 cases
2026-04-07 13:32:45,026 - SmartSOTA_Dynamic - INFO - Whole-brain val progress: 87/87 cases



Epoch 12: val_whole_dice_hard_raw did not improve from 0.00000
256/256 - 528s - 2s/step - dice_coefficient: 8.6670e-05 - loss: 1.1855 - safe_binary_iou: 0.2344 - val_dice_coefficient: 2.3747e-04 - val_whole_dice_micro: 3.1985e-04 - val_whole_dice_hard: 8.2411e-10 - val_whole_dice_hard_brainmask: 8.2411e-10 - val_whole_dice_hard_raw: 8.2411e-10 - val_whole_dice_hard_brainmask_delta: 0.0000e+00 - val_whole_dice_hard_postproc_delta: 0.0000e+00 - val_whole_dice_hard_thr_0p30: 8.2411e-10 - val_whole_dice_hard_thr_0p40: 8.2411e-10 - val_whole_dice_hard_thr_0p50: 8.2411e-10 - val_whole_dice_hard_thr_0p60: 8.2411e-10 - val_whole_dice_hard_thr_0p70: 8.2411e-10 - val_whole_dice_hard_raw_thr_0p30: 8.2411e-10 - val_whole_dice_hard_raw_thr_0p40: 8.2411e-10 - val_whole_dice_hard_raw_thr_0p50: 8.2411e-10 - val_whole_dice_hard_raw_thr_0p60: 8.2411e-10 - val_whole_dice_hard_raw_thr_0p70: 8.2411e-10 - val_whole_dice_hard_brainmask_thr_0p30: 8.2411e-10 - val_whole_dice_hard_brainmask_thr_0p40: 8.2411e-1

2026-04-07 13:32:45,338 - SmartSOTA_Dynamic - INFO - 📉 Loss mix @epoch 12: dice=0.400, boundary=0.600, bce=0.200, topk=0.000, volume=0.050, focal=0.000


Epoch 13/200


In [ ]:
# --------- Quick sanity prediction on zeros ---------
import numpy as np

cfg = seg.DynamicTrainingConfig(
    DATA_DIR=TRAIN_DIR,
    IMAGES_DIR=TRAIN_T1,
    MASKS_DIR=TRAIN_MASKS,
    INPUT_SHAPE=INPUT_SHAPE,
    PATCH_SIZE=PATCH_SIZE,
    MODEL_DIR=MODEL_DIR,
    CALLBACKS_DIR=CALLBACKS_DIR,
)

weights = CALLBACKS_DIR / "best_model_dynamic.weights.h5"
if weights.exists():
    m = seg.build_model_for_inference(cfg, weights_path=str(weights))
else:
    m = seg.build_model_for_inference(cfg)

x0 = np.zeros((1, *INPUT_SHAPE), np.float32)
p0 = m.predict(x0, verbose=0)[0, ..., 0]
print("Blank input -> p.mean=", float(p0.mean()), " p.max=", float(p0.max()))


Blank input -> p.mean= 0.10394287109375  p.max= 0.95703125
